# SVD: a Tiny Example

## Dot Product, Cosine Similarity and Projections

$$
v \cdot w = \sum_{i=1}^{n} v_i w_i
$$

$$
\cos(\theta) = \frac{v \cdot w}{||v|| ||w||} \quad \text{where } ||v|| = \sqrt{v\cdot v} \text{ is the length of } v
$$

## Singular Value Decomposition

$$
X = U \Sigma V^\top
$$

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

texts = [
    "bad",
    "nice name",
    "bad name name name",
    "nice"
]

X = CountVectorizer().fit_transform(texts).toarray()

pd.DataFrame(X, columns=CountVectorizer().fit(texts).get_feature_names_out(), index=texts)

In [ ]:
# Dot products: word similarities based on co-occurrence in documents
X.T @ X

## Similarities (dot products) and correlations between word vectors

In [ ]:
X1 = (X - X.mean(axis=0))
X1

## Eigenvalues and eigenvectors of word similarities

In [ ]:
def compute_eigen(X):
    eigenvalues, eigenvectors = np.linalg.eigh(X.T @ X)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    print("Eigenvalues:")
    print(eigenvalues.round(2))
    print("\nEigenvectors:")
    print(eigenvectors.round(2))

    return eigenvalues, eigenvectors

eigenvalues, eigenvectors = compute_eigen(X)

## Scoring a document against a direction in word space

In [ ]:
v = np.array([0.5, -0.5, 1])

X @ v

## The first eigenvector and maximal topic score

## Second singular value/vector

## Singular Layers

$$
U \Sigma V^\top = \sum_{i=1}^r \sigma_i u_i v_i^\top
$$

In [ ]:
# Reduced SVD: X = U @ diag(singular_values) @ Vt
U, singular_values, Vt = np.linalg.svd(X, full_matrices=False)

# Reconstruct X using the first 1, 2, and 3 singular components

rank = 3

X_rank = (U[:, :rank] * singular_values[:rank]) @ Vt[:rank, :]
print(f"\nRank-{rank} approximation:")
print(X_rank.round(2))

## Image example: building a picture from layers

In [ ]:
from PIL import Image
import requests
from io import BytesIO
import numpy as np

url = "https://raw.githubusercontent.com/febse/data/refs/heads/main/ta/example_photo.jpg"

response = requests.get(url)
img = Image.open(BytesIO(response.content)).convert("L")
img = img.resize((300, int(img.height * 300 / img.width)))
display(img)

In [ ]:
img_matrix = np.array(img)
print("Image matrix shape:", img_matrix.shape)

U_img, S_img, VT_img = np.linalg.svd(img_matrix, full_matrices=False)
print("U:", U_img.shape, " S:", S_img.shape, " V^T:", VT_img.shape)

print(img_matrix[:5, :5])  # Display a small portion of the image matrix for verification

In [ ]:
def show_reconstructed_image(start_layer, end_layer, U, S, VT):
    if not 0 <= start_layer < end_layer <= len(S):
        raise ValueError(f"Expected 0 <= start_layer < end_layer <= {len(S)}")

    # Reconstruct the image from the selected layers [start_layer, end_layer)
    range_reconstruction = (U[:, start_layer:end_layer] * S[start_layer:end_layer]) @ VT[start_layer:end_layer, :]

    range_uint8 = np.clip(range_reconstruction, 0, 255).astype(np.uint8)
    return Image.fromarray(range_uint8)

In [ ]:
# Reconstruct from layers [start_layer, end_layer); end_layer is excluded.
start_layer = 0
end_layer = 80

reconstructed_img = show_reconstructed_image(start_layer, end_layer, U_img, S_img, VT_img)
margin = 10
comparison = Image.new("L", (img.width + margin + reconstructed_img.width, img.height), color=255)
comparison.paste(img, (0, 0))
comparison.paste(reconstructed_img, (img.width + margin, 0))
display(comparison)